# Problema X33Y33Z33 — Método TFBGF em um Problema 3D (caso simulado)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x33y33z33_tfbgf.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição do problema

Paralelepípedo $0 \le x \le L$, $0 \le y \le W$, $0 \le z \le R$ representando uma ferramenta de corte, com:

- **convecção (tipo 3)** nas seis faces, com coeficiente $h$ e meio a $T_\infty$;
- um **fluxo de calor** $q(t)$ imposto numa **região retangular** da face superior, $x\in[L_1,L_2]$, $z\in[R_1,R_2]$ (a área de contato $a_p\times f$ do corte);
- **condição inicial** $T(x,y,z,0)=T_0=T_\infty$.

A notação **X33Y33Z33** indica convecção (3) em ambas as faces de cada uma das três direções. Este *notebook* demonstra, em um caso **simulado** (fluxo conhecido), o problema direto, a resposta impulsiva e a recuperação do fluxo pelo método TFBGF — a mesma máquina aplicada aos dados **experimentais** de torneamento na seção seguinte do livro.

## Solução analítica

Pelo **produto de funções de Green transientes** nas três direções, a temperatura devida ao fluxo imposto é uma **série tripla** nos autovalores $\alpha_m$ (direção $x$), $\beta_n$ (direção $y$) e $\gamma_p$ (direção $z$):

$$T(x,y,z,t) = T_\infty + \frac{\alpha}{k}\,\frac{8}{W}\sum_{m}\sum_{n}\sum_{p} \Phi_{mnp}(x,y,z)\,\big(q * e^{-A_{mnp} t}\big),\qquad
A_{mnp} = \Big[\big(\tfrac{\alpha_m}{L}\big)^2+\big(\tfrac{\beta_n}{W}\big)^2+\big(\tfrac{\gamma_p}{R}\big)^2\Big]\alpha .$$

Como $T_0=T_\infty$, a parcela da condição inicial se anula. A **resposta ao impulso** $h(x,y,z,t)$ tem a mesma estrutura espacial $\Phi_{mnp}$, com o fator temporal $e^{-A_{mnp}t}$ (sem a convolução com $q$).

### Autovalores

Em cada direção, a convecção nas duas faces leva à equação transcendental

$$(\gamma^2 - B_a B_b)\sin\gamma - \gamma\,(B_a+B_b)\cos\gamma = 0,$$

com os números de Biot $B_a$, $B_b$ das duas faces. As raízes são obtidas numericamente pelo **método de Brent**.

## Bibliotecas utilizadas

- **NumPy** (`numpy`): computação numérica, séries triplas vetorizadas (`np.einsum`/produto externo) e a **FFT** (`numpy.fft`) da deconvolução.
- **Matplotlib** (`matplotlib.pyplot`): gráficos.
- **SciPy** (`scipy.optimize.brentq`): método de Brent para os autovalores.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq

## Parâmetros do problema

Propriedades do **aço rápido** da ferramenta e geometria aproximada (as mesmas do caso experimental do livro). O fluxo desconhecido a recuperar é um **pulso triangular** imposto na janela de corte de $10$ a $60$ s.

In [ ]:
k, alpha = 24.0, 7.0868e-6           # condutividade [W/(m·K)] e difusividade [m²/s] do aço rápido
L, W, Rc = 12.7e-3, 12.7e-3, 101.6e-3  # dimensões em x, y, z [m]
h        = 100.0                     # coeficiente convectivo em todas as faces [W/(m²·K)]
T0 = Tinf = 26.73                    # temperatura inicial = ambiente [°C]
dt, tf   = 0.3, 120.0                # discretização do tempo [s]
M        = 10                        # número de termos da série em cada direção

# área de imposição do fluxo na face superior (a_p × f = 1,9 mm²)
L1, L2, R1, R2 = 0.0, 2.0e-3, 0.0, 0.95e-3

t = np.arange(0.0, tf + dt/2, dt)
N = len(t)

# pulso triangular de fluxo de calor [W/m²]
qpico, t0, tpico, tbase = 2.0e6, 10.0, 35.0, 60.0
q = np.where(t < t0, 0.0,
    np.where(t <= tpico, qpico*(t-t0)/(tpico-t0),
    np.where(t <= tbase, qpico*(tbase-t)/(tbase-tpico), 0.0)))

# números de Biot (iguais nas duas faces de cada direção, pois h é o mesmo)
B1 = h*L/k;  B3 = h*W/k;  B5 = h*Rc/k
print(f'Biot:  Bx = {B1:.4f},  By = {B3:.4f},  Bz = {B5:.4f}')

## Autovalores das três direções

Cada raiz $\gamma$ da equação transcendental é buscada no seu ramo por `brentq`, varrendo o eixo em passos de $\pi/400$ e aplicando o método de Brent sempre que a função troca de sinal.

In [ ]:
def autovalores(n, Ba, Bb):
    """n menores raízes positivas de (γ²-Ba·Bb)·sin γ - γ(Ba+Bb)·cos γ = 0."""
    f = lambda g: (g*g - Ba*Bb)*np.sin(g) - g*(Ba+Bb)*np.cos(g)
    raizes, g, passo = [], 1e-6, np.pi/400
    while len(raizes) < n and g < (n + 3)*np.pi:
        g2 = g + passo
        if f(g)*f(g2) < 0:
            raizes.append(brentq(f, g, g2))
        g = g2
    return np.array(raizes)

alfam = autovalores(M, B1, B1)   # direção x
betan = autovalores(M, B3, B3)   # direção y
gamap = autovalores(M, B5, B5)   # direção z
print('α_m[:3] =', np.round(alfam[:3], 5))
print('β_n[:3] =', np.round(betan[:3], 5))
print('γ_p[:3] =', np.round(gamap[:3], 5))

## Problema direto e resposta impulsiva

Monta-se o coeficiente espacial $\Phi_{mnp}$ (produto das três direções) e o autovalor combinado $A_{mnp}$. A temperatura integra o fluxo discreto por uma **recorrência estável** (convolução exata em cada passo); a resposta impulsiva é a mesma série com o fator $e^{-A_{mnp}t}$.

In [ ]:
def _coef(e, B, coord, D):
    """Coeficiente espacial de uma direção (autofunção normalizada) na posição coord."""
    return (e*np.cos(e*coord/D) + B*np.sin(e*coord/D)) / ((e**2+B**2)*(1 + B/(e**2+B**2)) + B)

# autovalor combinado A_mnp (achatado em um vetor de M³ modos)
Amnp = (((alfam[:, None, None]/L)**2
       + (betan[None, :, None]/W)**2
       + (gamap[None, None, :]/Rc)**2) * alpha).ravel()
decai = np.exp(-Amnp*dt)
onem  = 1.0 - decai

def _phi(x, y, z):
    """Coeficiente espacial Φ_mnp do termo de fluxo, em (x,y,z), achatado."""
    Cx = _coef(alfam, B1, x, L); Cy = _coef(betan, B3, y, W); Cz = _coef(gamap, B5, z, Rc)
    Fx = (np.sin(alfam*L2/L) - np.sin(alfam*L1/L)
          - (B1/alfam)*np.cos(alfam*L2/L) + (B1/alfam)*np.cos(alfam*L1/L))
    Fz = (np.sin(gamap*R2/Rc) - np.sin(gamap*R1/Rc)
          - (B5/gamap)*np.cos(gamap*R2/Rc) + (B5/gamap)*np.cos(gamap*R1/Rc))
    Ax = Cx*Fx; Ay = Cy*(betan*np.cos(betan) + B3*np.sin(betan)); Az = Cz*Fz
    return (Ax[:, None, None]*Ay[None, :, None]*Az[None, None, :]).ravel()

def temperatura(x, y, z, q):
    """T(x,y,z,t) do problema direto X33Y33Z33 (T0 = T∞)."""
    coef = _phi(x, y, z) / Amnp
    J = np.zeros(len(Amnp)); T = np.zeros(N)
    for d in range(N):
        J = decai*J + q[d]                      # recorrência da convolução
        T[d] = (alpha/k)*(8/W)*(coef @ (onem*J))
    return Tinf + T

def resposta_impulsiva(x, y, z):
    """h(x,y,z,t) do problema X33Y33Z33 (h em t=0 fixado em 0)."""
    A = _phi(x, y, z); H = np.zeros(N)
    H[1:] = (alpha/k)*(8/W)*(A @ np.exp(-Amnp[:, None]*t[None, 1:]))
    return H

# posição de análise (próxima da fonte de calor)
xp, yp, zp = 9.28e-3, W, 3.03e-3
T_sim = temperatura(xp, yp, zp, q)
H_sim = resposta_impulsiva(xp, yp, zp)
print(f'T: de {T_sim.min():.2f} a {T_sim.max():.2f} °C   |   h(∞) tende a {H_sim[-1]:.2e} m²·K/J')

## Temperatura e resposta impulsiva

A temperatura sobe durante o corte e volta a cair pela convecção; a resposta impulsiva parte de um pico e decai lentamente (sem retornar a zero dentro da janela).

In [ ]:
cores = ['#D55E00', '#0072B2', '#009E73']   # paleta Okabe-Ito

fig, (axa, axb) = plt.subplots(1, 2, figsize=(10, 4))
axa.plot(t, T_sim, color=cores[0], linewidth=1.6)
axa.set_xlabel('Tempo (s)'); axa.set_ylabel('T(x, y, z, t)  (°C)')
axa.set_title('Temperatura simulada'); axa.grid(True)

axb.plot(t[1:], H_sim[1:], color=cores[1], linewidth=1.6)
axb.set_xlabel('Tempo (s)'); axb.set_ylabel('h(x, y, z, t)  (m²·K/J)')
axb.set_title('Resposta impulsiva'); axb.grid(True)
plt.tight_layout(); plt.show()

## Problema inverso: deconvolução das derivadas

Como $h$ não retorna a zero dentro da janela de observação (a difusão na peça grande é lenta), a deconvolução direta via FFT sofre vazamento espectral. Faz-se então a deconvolução sobre as **derivadas temporais** dos sinais, $dT/dt = q * (dh/dt)$, que decaem a zero — a mesma técnica usada no problema 1D X22. O pulso triangular é recuperado com erro de área muito inferior a $1\%$.

In [ ]:
NR = 2**20
freqs = np.fft.fftfreq(NR, dt)

def estima_fluxo(T, H, fc=None, ordem=4):
    """q(t) por deconvolução de dT/dt com dh/dt; filtro passa-baixa opcional."""
    Wf = 1.0 if fc is None else 1.0/(1.0 + (np.abs(freqs)/fc)**(2*ordem))
    qf = np.fft.fft(np.diff(T), NR) / np.fft.fft(np.diff(H), NR) * Wf
    return np.real(np.fft.ifft(qf))[:N] / dt

q_est = estima_fluxo(T_sim, H_sim)
corte = N - 15                       # descarta a borda final (efeito da FFT)

fig, ax = plt.subplots()
ax.plot(t[:corte], q[:corte]/1e6, color='k', linewidth=1.8, label='q imposto')
ax.plot(t[:corte:6], q_est[:corte:6]/1e6, color=cores[0], linestyle='none',
        marker='o', markersize=4, markerfacecolor='none', label='q estimado')
ax.set_xlabel('Tempo (s)'); ax.set_ylabel('q(t)  ($\\times 10^{6}$ W/m²)')
ax.set_title('Fluxo de calor imposto e estimado (X33Y33Z33)')
ax.set_xlim(0, tf); ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

A1 = np.trapezoid(q[:corte], t[:corte])
A2 = np.trapezoid(q_est[:corte], t[:corte])
print(f'Área imposta A1 = {A1:.4e} W·s/m²;  estimada A2 = {A2:.4e};  erro = {100*(A2-A1)/A1:+.3f} %')

## Verificação: temperatura reconstruída

Realimentando o fluxo estimado na solução direta, a temperatura resultante reproduz a simulada — confirmação de que o fluxo recuperado descreve o sistema.

In [ ]:
T_rec = temperatura(xp, yp, zp, q_est)

fig, ax = plt.subplots()
ax.plot(t[:corte], T_sim[:corte], color='k', linewidth=1.5, label='T simulada')
ax.plot(t[:corte:8], T_rec[:corte:8], color=cores[0], linestyle='none', marker='o',
        markersize=4.5, markerfacecolor='none', label='T reconstruída (via q estimado)')
ax.set_xlabel('Tempo (s)'); ax.set_ylabel('T(x, y, z, t)  (°C)')
ax.set_title('Verificação da estimativa 3D'); ax.set_xlim(0, tf)
ax.legend(); ax.grid(True)
plt.tight_layout(); plt.show()

erro = np.mean(np.abs(T_rec[3:corte] - T_sim[3:corte]))
print(f'Erro médio |T_reconstruída - T_simulada| = {erro:.4f} °C')